# ManualChaidTree — manual demo

Hand-build a CHAID scorecard tree on the Kaggle **Default of Credit Card Clients** dataset.

1. Download the dataset from https://www.kaggle.com/datasets/uciml/default-of-credit-card-clients-dataset
2. Place `UCI_Credit_Card.csv` in the project root (one folder above this notebook).
3. Run the cells below.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from sklearn.model_selection import train_test_split
from manual_chaid import ManualChaidTree

df = pd.read_csv('../UCI_Credit_Card.csv').drop(columns=['ID'])
y = df.pop('default.payment.next.month')
X = df

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
X_train = X_train.reset_index(drop=True); y_train = y_train.reset_index(drop=True)
X_test  = X_test.reset_index(drop=True);  y_test  = y_test.reset_index(drop=True)
print(X_train.shape, 'target rate =', round(y_train.mean(), 4))
X_train.head()

(22500, 23) target rate = 0.2212


,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6
0,200000.0,1,1,2,32,-1,-1,-1,-2,-2,...,0.0,0.0,0.0,0.0,2182.0,0.0,0.0,0.0,0.0,0.0
1,320000.0,2,1,2,28,0,0,0,0,0,...,290057.0,25922.0,22982.0,20553.0,13300.0,12756.0,1295.0,10312.0,8394.0,8242.0
2,170000.0,2,1,2,27,-1,-1,2,-1,-1,...,736.0,736.0,736.0,736.0,1536.0,0.0,736.0,736.0,736.0,736.0
3,90000.0,2,3,1,40,0,0,0,0,0,...,76947.0,46589.0,47730.0,49488.0,3416.0,4142.0,1700.0,1888.0,2710.0,1500.0
4,250000.0,2,1,1,43,-1,-1,-1,0,0,...,45297.0,17371.0,11481.0,5922.0,24890.0,48394.0,0.0,5461.0,15000.0,6000.0


## 1. Launch the interactive builder

`train()` opens the tkinter window (canvas + side menu). `PAY_0..PAY_6` are repayment-status
columns where `0` and `-1` carry special meaning, so we pre-separate them.

Build the tree in the window: pick a leaf, choose a high-χ² variable, tweak cut-offs, **Split**;
then **Combine** / **Prune** / **Undo** as needed; finally **Save** and **Export** from the Actions panel.
Close the window to return here.

In [ ]:
tree = ManualChaidTree(X_train, y_train)
tree.train(
    # pre_separate_zero=['PAY_0', 'PAY_2', 'PAY_3'],
    # pre_separate_values={'PAY_0': -1, 'PAY_2': -1},
    min_samples=200,
    max_children=9,
)

## 2. (Optional) Build headlessly to test the core without the UI

Everything the UI does is available on the object. Pass `launch_ui=False` and drive it from code.

In [ ]:
def show_tree(t):
    for nid in sorted(t.nodes):
        n = t.nodes[nid]
        pad = '  ' * n.depth
        rule = n.condition.describe() if n.condition else 'ROOT'
        extra = ''
        if not n.is_leaf:
            extra = f"  | split={n.split_variable} chi2={n.split_chi2:.1f} p={n.split_pvalue:.2g}"
        print(f"{pad}#{nid} [{rule}] n={n.n_samples} bad={n.target_rate:.1%}{extra}")

t = ManualChaidTree(X_train, y_train)
t.train(pre_separate_zero=['PAY_0'], pre_separate_values={'PAY_0': -1},
        min_samples=500, max_children=4, launch_ui=False)

# Rank candidate variables for the root by chi-square:
cands = t.candidate_splits(0)
for c in cands[:6]:
    print(f"{c['variable']:<10} chi2={c['chi2']:.1f}  p={c['pvalue']:.2g}  splittable={c['splittable']}")

In [ ]:
# Split the root on the top candidate, then split a child, then inspect.
top = cands[0]
t.split(0, top['variable'], top['conditions'])

leaf = max(t.leaves(), key=lambda n: n.n_samples)
child_cands = t.candidate_splits(leaf.node_id)
best = next((c for c in child_cands if c['splittable']), None)
if best:
    t.split(leaf.node_id, best['variable'], best['conditions'])

show_tree(t)
print('\nhistory:', t.history)

In [ ]:
# Combine / prune / undo round-trip
root_children = t.nodes[0].children
if len(root_children) >= 2:
    leaf_kids = [c for c in root_children if t.nodes[c].is_leaf][:2]
    if len(leaf_kids) == 2:
        t.combine(0, leaf_kids)
        print('after combine:', t.history[-1])
        t.undo()
        print('after undo, history:', t.history)

## 3. Gini, save / load, and scoring new data

In [ ]:
X_train[(X_train['PAY_0'] != 0) & (X_train['PAY_0'] != -1) & (X_train['PAY_0'] < 1.5) & (X_train['PAY_0'] >= -0.5)].shape

# 13,844

In [10]:
# Reload WITH data -> editable; WITHOUT data -> view / score only.
t_edit = ManualChaidTree.load('tree.pkl', X=X_train, y=y_train)
print('reloaded (with data) train Gini:', round(t_edit.gini(), 4))

t_view = ManualChaidTree.load('tree.pkl')
print('scored new rows (no data needed):', t_view.predict_proba(X_test.head()))

reloaded (with data) train Gini: 0.5553
scored new rows (no data needed): [0.37102473 0.37102473 0.32370638 0.49666222 0.18165468]


In [11]:
t_edit.train(pre_separate_values={'PAY_0': -1}, min_samples=500, max_children=4)

In [12]:
t_edit.depth()

5

In [ ]:
t_view.train()

In [7]:
t_view.save_png('tree4.png')

'tree4.png'

In [13]:
t_view.predict_proba(X_test.head())

array([0.37102473, 0.37102473, 0.32370638, 0.49666222, 0.18165468])

In [14]:
t_view.predict_node(X_test.head())

array([43, 43, 40, 45, 54])

## 4. Export the leaf rule set to Excel

In [ ]:
rules = t.export_rules_excel('leaf_rules.xlsx')
rules